# Part 5 — Fetching Data from APIs, and Loading CSV/JSON Files
### Data Mondays, Week 4 — Amani Insurance claims case study

**Deliverable:** fetch data from **APIs** and load **CSV** or **JSON** files.

**Business scenario:** Amani Insurance cedes (shares) a portion of its large
property and motor-theft claims with an international reinsurer who prices
everything in USD. To report ceded claims correctly, Amani needs the current
USD/KES exchange rate.

This notebook:
1. Loads the claims **CSV** (familiar by now) and a **JSON** policyholder master file (new).
2. Fetches a **live** USD/KES exchange rate from a free, keyless public API.
3. Falls back to a small **cached JSON file** automatically if there's no internet.
4. Joins claims + policyholders + exchange rate into one view, and reports ceded
   claims in both KES and USD.

Requires the `requests` library (`pip install requests`).

In [ ]:
import json

import pandas as pd
import requests

CLAIMS_PATH = "insurance_claims_messy.csv"
POLICYHOLDERS_PATH = "policyholders.json"
FALLBACK_RATES_PATH = "exchange_rates_cache.json"

API_URL = "https://api.frankfurter.app/latest"
# NOTE: if your network blocks this URL, or there's no wifi, the fallback
# further down kicks in automatically -- the notebook still runs either way.

## Section 1 — Loading a CSV (recap) and a JSON file (new)

Two ways to load JSON are shown here, because you'll see both patterns in real
code: `json.load` when you need to inspect or transform the structure first,
`pd.read_json` when the JSON is already a flat list of records and you just
want a DataFrame.

In [ ]:
claims = pd.read_csv(CLAIMS_PATH)
print(f"claims: {claims.shape[0]} rows from CSV")

In [ ]:
# json.load() reads the file into plain Python objects -- here, a list of dicts.
with open(POLICYHOLDERS_PATH, "r", encoding="utf-8") as f:
    policyholders_raw = json.load(f)

print(f"policyholders_raw is a {type(policyholders_raw).__name__} of {len(policyholders_raw)} dicts.")
print("First record:")
policyholders_raw[0]

In [ ]:
# pd.read_json() skips the intermediate step and goes straight to a DataFrame,
# since this JSON file is already a flat list of same-shaped records.
policyholders = pd.read_json(POLICYHOLDERS_PATH)
print(f"policyholders DataFrame: {policyholders.shape}")
policyholders.head(3)

In [ ]:
# Even a "system-exported" file isn't always perfect -- check for gaps here too.
print("Missing values in policyholders (system export, still not perfect):")
policyholders.isna().sum()

## Section 2 — Fetching live data from a public API

An **API call** here is just an HTTP request that returns data (usually JSON)
instead of a web page. We wrap it in a function that tries the live call first,
and automatically falls back to a cached file if anything goes wrong — this
"try live, fall back to cache" pattern is exactly what production systems do
too, not just classroom demos.

In [ ]:
def fetch_live_usd_kes_rate():
    """GET request to api.frankfurter.app, a free/keyless currency API.
    Returns a float rate, or raises an exception if the call fails
    (no internet, API down, etc.) -- the caller decides what to do next.
    """
    response = requests.get(API_URL, params={"from": "USD", "to": "KES"}, timeout=5)
    response.raise_for_status()   # turns a bad HTTP status (e.g. 404, 500) into an exception
    data = response.json()        # parse the JSON response body into a Python dict
    print("Raw API response:", data)
    return data["rates"]["KES"]

In [ ]:
def get_usd_kes_rate():
    """Try the live API first; fall back to the cached JSON file if
    anything goes wrong."""
    try:
        rate = fetch_live_usd_kes_rate()
        print(f"Using LIVE rate: 1 USD = {rate} KES")
        return rate, "live"
    except (requests.RequestException, KeyError, ValueError) as e:
        # requests.RequestException catches network/HTTP problems;
        # KeyError/ValueError catch a malformed or unexpected response body.
        print(f"Live API call failed ({e.__class__.__name__}: {e}). Falling back to cached rate.")
        with open(FALLBACK_RATES_PATH, "r", encoding="utf-8") as f:
            cache = json.load(f)
        most_recent_date = max(cache["rates_by_date"])   # the latest cached date
        rate = cache["rates_by_date"][most_recent_date]
        print(f"Using CACHED rate from {most_recent_date}: 1 USD = {rate} KES ({cache['note']})")
        return rate, "cached"


usd_kes_rate, rate_source = get_usd_kes_rate()

**What this pattern buys us:** `get_usd_kes_rate()` never crashes the whole
notebook just because the wifi is down or the API is temporarily unavailable —
that's the difference between a demo that only works when everything's perfect,
and code you could actually ship.

## Section 3 — Joining claims + policyholders, converting to USD

Now bring everything together: clean the claims amounts, join in the
policyholder details by `policy_number`, convert to USD using the rate from
Section 2, and report the claims that get ceded (shared) with the reinsurer.

In [ ]:
# Same cleaning pattern as Parts 3 & 4.
claims["claim_amount_kes"] = (
    claims["claim_amount_kes"].astype(str).str.replace(",", "", regex=False).str.strip()
)
claims["claim_amount_kes"] = pd.to_numeric(claims["claim_amount_kes"], errors="coerce").abs()
claims["claim_type"] = claims["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
claims["region"] = claims["region"].astype(str).str.strip().str.title()
group_median = claims.groupby("claim_type")["claim_amount_kes"].transform("median")
claims["claim_amount_kes"] = claims["claim_amount_kes"].fillna(group_median)

In [ ]:
# Join claims to policyholder master data on policy_number -- this is why
# loading BOTH sources mattered: the claims log alone doesn't know a
# policyholder's cover type or sum insured.
merged = claims.merge(policyholders, on="policy_number", how="left", suffixes=("", "_policy"))
print(f"Merged shape: {merged.shape}  "
      f"(claims: {claims.shape[0]}, policyholders matched: {merged['full_name'].notna().sum()})")

merged["claim_amount_usd"] = merged["claim_amount_kes"] / usd_kes_rate

In [ ]:
# Reinsurance treaty: anything over KES 500,000 gets ceded (shared) with
# the reinsurer -- report those specifically, in both currencies.
CESSION_THRESHOLD_KES = 500_000
ceded = merged[merged["claim_amount_kes"] > CESSION_THRESHOLD_KES].copy()
ceded = ceded.sort_values("claim_amount_kes", ascending=False)

print(f"Ceded claims (> KES {CESSION_THRESHOLD_KES:,}): {len(ceded)}")
print(f"Exchange rate used: 1 USD = {usd_kes_rate} KES (source: {rate_source})")

In [ ]:
ceded[["claim_id", "claim_type", "region", "claim_amount_kes", "claim_amount_usd"]].head(5).style.format({
    "claim_amount_kes": "{:,.0f}",
    "claim_amount_usd": "{:,.0f}",
})

In [ ]:
total_ceded_usd = ceded["claim_amount_usd"].sum()
print(f"Total ceded exposure: USD {total_ceded_usd:,.0f}")

**What we just built:** one working DataFrame assembled from three different
kinds of source — a local CSV, a local JSON file, and a live web API — with a
safety net so a network hiccup never breaks the whole analysis.

**Up next:** `06_student_exercise.ipynb` gives you one hands-on task per
deliverable from this week, so you can practice each pattern yourself.